In [5]:
import os

# Disable progress bars for Hugging Face Hub and tqdm globally
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"

In [6]:
from transformers.utils import logging
logging.set_verbosity_error()

In [7]:
import logging
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()

In [8]:
#required libraries installation
!pip install transformers torch --quiet

In [10]:
#importing the libraries
import warnings
warnings.filterwarnings('ignore') # Suppress unnecessary warnings

# Hugging Face Transformers imports
from transformers import AutoModelForCausalLM, AutoTokenizer

# PyTorch - used as the backend for model inference
import torch

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

All libraries imported successfully!
PyTorch version: 2.11.0+cpu


In [11]:
#Loading the pre-trained model
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading tokenizer for '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"⏳ Loading model for '{MODEL_NAME}'... (this may take a minute)")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode - disables dropout layers used only during training
model.eval()

print("\nModel and Tokenizer loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

⏳ Loading tokenizer for 'microsoft/DialoGPT-medium'...
⏳ Loading model for 'microsoft/DialoGPT-medium'... (this may take a minute)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


Model and Tokenizer loaded successfully!
Model parameters: 354,823,168


In [12]:
chat_history_ids = None

In [14]:
#building the chatbot loop
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye! Have a great day.")
        break
        
    # Encode user input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")
    
    # Append to chat history
    if chat_history_ids is not None:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        input_ids = new_input_ids
        
    # Generate response
    chat_history_ids = model.generate(
    input_ids,
    max_length=1000,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,          # Allows for more creative answers
    top_k=50,                # Keeps the top 50 most likely words
    top_p=0.95,              # Uses "nucleus sampling"
    temperature=0.7,         # Makes the output more or less random (1.0 is default)
    repetition_penalty=1.2   # Strictly discourages repeating the same phrase
)
    
    # Decode response
    # (Note: The image cuts off the decoding part, but this is the standard completion)
    response = tokenizer.decode(chat_history_ids[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    print(f"Chatbot: {response}")

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.



You:  Hello,How is your day going


Chatbot: It was okay, i guess?


You:  iam feeling a bit tired today


Chatbot: Hey man, I am glad to have you back!


You:  Did you see that movie Interstellar?


Chatbot: Oh yeah, that one was great. I hope your day went well though!


You:  quit


Chatbot: Goodbye! Have a great day.
